# Phase 5 — confirmatory test on models not seen before

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/phase5_confirmatory.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**What Phase 4 found (exploratory).** Moving each image by $\pm\delta u_1$, where $u_1$ is the top eigenvector of
the clean input-gradient covariance $C$, raises $\lambda_1$ sharply for some models and hardly at all for others. The
amplification is

$$A = \lambda_1(C_\delta) / \lambda_1(C), \qquad C_\delta = \mathbb{E}[g(x \pm \delta u_1) g(x \pm \delta u_1)^T],\quad \delta = 4.$$

In Phase 4, $A$ was about ×21 for base models and ×14 for a weak gradient penalty, but only about ×3–5 for
adversarially trained models. A random shift of the same norm gave ×1. The shifted $\lambda_1(C_\delta)$ also
followed universal fooling better than the clean $\lambda_1$. These patterns were found after looking at the data,
so here they are tested on **models that were not used to find them.**

**Held-out groups.**
1. **OrganAMNIST, VGG-19** (new dataset, new models): base, AS fine-tuning, UAT, PGD-AT, trace penalty β = 0.1 and β = 1.
2. **OrganAMNIST, ResNet-18**: the same six models.
3. **BloodMNIST, VGG-19, the 12 models of Phases 1 and 2.** These are separate training runs from Phase 3: base, noise,
   AS / UAP fine-tuning, UAT, PGD-AT, and the six $\lambda_1$ / trace penalty models.

Adversarially trained models are AS fine-tuning, UAP fine-tuning, UAT and PGD-AT. All other models are
non-adversarial: base, noise, and the gradient or spectral penalties.

**Rules, written before the run.**
- **C1, mechanism.** In each of the 3 groups, the median $A$ of the adversarially trained models is lower than the
  median $A$ of the non-adversarial models. All 3 groups are required.
- **C2, prediction.** The within-group Spearman ρ between $\lambda_1(C_\delta)$ and AS fooling is ≥ 0.7, and higher
  than the clean-$\lambda_1$ ρ, in at least 2 of the 3 groups.
- The Phase 4 finding is **confirmed** if C1 and C2 both hold. Otherwise it is **not confirmed**, and is reported
  as an exploratory observation only.
- Also reported: a pooled one-sided Mann–Whitney test of $A$ (adversarial < non-adversarial), and the random-shift control.

**Run.** Use the Google account that holds `active_subspace_runs` (the Phase 1–2 models are read from there).
Run the first cell, then the second (press **Allow**), then *Run all* with a T4 GPU. It takes about 4 hours. Each
model and each row is saved in `active_subspace_runs/phase5` as soon as it is ready, and *Run all* again continues
after a disconnect.

In [ ]:
%cd /content
!rm -rf active-subspace-bloodmnist && git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git log --oneline -1
!pip install -q medmnist

In [ ]:
import os, copy, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu

from asnet.data import CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import sgd_uap
from asnet.robust import adv_finetune, uat_finetune, pgd_finetune, spectral_finetune, as_perturbation
from asnet.spectrum import input_spectrum, shifted_spectrum

ROOT = "/content/drive/MyDrive/active_subspace_runs"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is NOT connected. Run this cell again and press Allow.")
else:
    ROOT = "runs"
OUT = os.path.join(ROOT, "phase5")
MODELS = os.path.join(OUT, "models")
os.makedirs(MODELS, exist_ok=True)
CSV = os.path.join(OUT, "phase5_models.csv")
device = get_device()
torch.backends.cudnn.benchmark = True
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("saving to", OUT, "|", device)

## Settings (the same as Phases 3 and 4)

In [ ]:
SIZE, EPOCHS, FT_EPOCHS, MAX_TRAIN = 64, 20, 5, 12000
DELTA, REFRESH, PGD_EPS = 4, 60, 2.0
N_ATTACK, N_SPECTRUM = 200, 2000
NEW_DATASET, ARCHS = "organamnist", ["vgg19", "resnet18"]
DEFENSES = ["AS fine-tuning", "UAT", "PGD-AT", "trace beta=0.1", "trace beta=1"]
ADVERSARIAL = {"AS fine-tuning", "UAP fine-tuning", "UAT", "PGD-AT"}

## Helpers

In [ ]:
_data = {}
def data(name):
    if name not in _data:
        train_aug, _, _ = get_datasets(SIZE, augment=True, name=name)
        train, val, test = get_datasets(SIZE, name=name)
        if len(train) > MAX_TRAIN:
            train_aug, train = subset(train_aug, MAX_TRAIN, seed=7), subset(train, MAX_TRAIN, seed=7)
        Xte, yte = stack_dataset(test, device=device)
        _data[name] = dict(train=train, val=val, Xte=Xte, yte=yte,
                           loader=make_loader(train_aug, 128, shuffle=True),
                           spec=make_loader(subset(train, N_SPECTRUM, seed=10), 100))
    return _data[name]

def ckpt(*parts):
    return os.path.join(MODELS, "_".join(str(p) for p in parts).replace(" ", "").replace("=", "") + ".pt")

def load_or_train(path, arch, nc, train_fn):
    m = build_model(arch, nc, SIZE)
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu"); m.load_state_dict(ck["state_dict"]); t = ck["time"]
    else:
        t0 = time.time(); m = train_fn(m); t = time.time() - t0
        torch.save({"state_dict": m.state_dict(), "time": t}, path)
    return m.to(device).eval(), t

def spectra(m, loader):
    for p in m.parameters():
        p.requires_grad_(False)
    clean = input_spectrum(m, loader, device=device, return_top_vector=True)
    act = shifted_spectrum(m, loader, DELTA, "active", u1=clean["u1"], device=device)
    rnd = shifted_spectrum(m, loader, DELTA, "random", device=device)
    for p in m.parameters():
        p.requires_grad_(True)
    return {"lambda1 clean": clean["lambda1"], "trace clean": clean["trace"],
            "lambda1 active d4": act["lambda1"], "trace active d4": act["trace"],
            "lambda1 random d4": rnd["lambda1"],
            "A (active)": act["lambda1"] / clean["lambda1"], "A (random)": rnd["lambda1"] / clean["lambda1"]}

def attack(m, d, nc, seed=0):
    for p in m.parameters():
        p.requires_grad_(False)
    pred = predict(m, d["Xte"])
    per_class = [(pred[d["yte"] == c] == c).float().mean().item() for c in range(nc) if (d["yte"] == c).any()]
    Xs, ys = stack_dataset(subset(d["train"], N_ATTACK, seed=500 + seed), device=device)
    row = {"clean acc": (pred == d["yte"]).float().mean().item(), "balanced acc": float(np.mean(per_class)),
           "AS fooling": fooling_rate(m, d["Xte"], as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed, log=lambda *a: None).v),
           "SGD-UAP fooling": fooling_rate(m, d["Xte"], sgd_uap(m, Xs, ys, delta=DELTA, seed=seed))}
    for p in m.parameters():
        p.requires_grad_(True)
    return row

rows = pd.read_csv(CSV).to_dict("records") if os.path.exists(CSV) else []
done = {(r["group"], r["model"]) for r in rows}
def append(row):
    rows.append(row); pd.DataFrame(rows).to_csv(CSV, index=False)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

## Groups 1 and 2 — OrganAMNIST, new models (about 4 hours)

In [ ]:
d, nc = data(NEW_DATASET), CLASSES[NEW_DATASET]
for arch in ARCHS:
    group = f"organamnist {arch}"
    print("=====", group)
    def train_base(m):
        set_seed(0)
        return train_classifier(m, d["loader"], make_loader(d["val"], 256), epochs=EPOCHS, device=device, log=lambda *a: None)[0]
    base, t = load_or_train(ckpt(NEW_DATASET, arch, "base"), arch, nc, train_base)
    if (group, "none (base)") not in done:
        append({"group": group, "model": "none (base)", "adversarial": False, "train time (s)": t,
                **attack(base, d, nc), **spectra(base, d["spec"])})
    Xa, ya = stack_dataset(subset(d["train"], 100, seed=300), device=device)
    for name in DEFENSES:
        if (group, name) in done:
            print(name, "already done"); continue
        def train_fn(_, name=name):
            set_seed(1); m = copy.deepcopy(base)
            if name == "AS fine-tuning":
                return adv_finetune(m, d["loader"], as_perturbation(Xa, ya, DELTA), epochs=FT_EPOCHS, refresh=REFRESH, device=device, log=lambda *a: None)[0]
            if name == "UAT":
                return uat_finetune(m, d["loader"], DELTA, epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
            if name == "PGD-AT":
                return pgd_finetune(m, d["loader"], PGD_EPS, epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
            return spectral_finetune(m, d["loader"], float(name.split("=")[1]), mode="trace", epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
        m, t = load_or_train(ckpt(NEW_DATASET, arch, name), arch, nc, train_fn)
        append({"group": group, "model": name, "adversarial": name in ADVERSARIAL, "train time (s)": t,
                **attack(m, d, nc), **spectra(m, d["spec"])})
        del m; torch.cuda.empty_cache()
    del base; torch.cuda.empty_cache()

## Group 3 — the 12 BloodMNIST models of Phases 1 and 2 (only measured, about 10 min)

Their AS fooling rates are read from the Phase 1 and 2 result files.

In [ ]:
group = "bloodmnist vgg19 (phases 1-2)"
p1 = pd.read_csv(os.path.join(ROOT, "phase1_spectrum.csv"))
p2 = pd.read_csv(os.path.join(ROOT, "phase2_regularization.csv"))
old = [(r["model"], r["AS fooling"], r["clean acc"]) for _, r in p1.iterrows()] + \
      [(r["model"], r["AS fooling"], r["clean acc"]) for _, r in p2.iterrows()]

def old_path(name):
    if name == "none (base)":
        return os.path.join(ROOT, "vgg19_bloodmnist.pt")
    if name.startswith("lambda1 beta=") or name.startswith("trace beta="):
        mode, beta = name.split(" beta=")
        return os.path.join(ROOT, "models", f"phase2_{mode}_beta{float(beta):g}.pt")
    return os.path.join(ROOT, "models", name.replace(" ", "_").replace("(", "").replace(")", "") + ".pt")

blood = data("bloodmnist")
for name, fool, acc in old:
    if (group, name) in done:
        continue
    path = old_path(name)
    if not os.path.exists(path):
        print("MISSING", path); continue
    ck = torch.load(path, map_location="cpu")
    m = build_model("vgg19", CLASSES["bloodmnist"], SIZE)
    m.load_state_dict(ck["state_dict"] if "state_dict" in ck else ck)
    m.to(device).eval()
    append({"group": group, "model": name, "adversarial": name in ADVERSARIAL, "train time (s)": np.nan,
            "clean acc": acc, "AS fooling": fool, **spectra(m, blood["spec"])})
    del m; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res[["group", "model", "adversarial", "clean acc", "AS fooling", "lambda1 clean", "lambda1 active d4", "A (active)", "A (random)"]].round(3)

## The decision

In [ ]:
lines, table = [], []
c1, c2 = 0, 0
for grp_name, g in res.groupby("group"):
    adv, non = g[g.adversarial.astype(bool)], g[~g.adversarial.astype(bool)]
    r_shift = spearmanr(g["lambda1 active d4"], g["AS fooling"])[0]
    r_clean = spearmanr(g["lambda1 clean"], g["AS fooling"])[0]
    ok1 = adv["A (active)"].median() < non["A (active)"].median()
    ok2 = r_shift >= 0.7 and r_shift > r_clean
    c1 += ok1; c2 += ok2
    table.append({"group": grp_name, "n": len(g), "median A adversarial": adv["A (active)"].median(),
                  "median A non-adversarial": non["A (active)"].median(), "C1 ok": ok1,
                  "rho shifted": r_shift, "rho clean": r_clean, "C2 ok": ok2,
                  "median A random": g["A (random)"].median()})
table = pd.DataFrame(table)
table.to_csv(os.path.join(OUT, "phase5_groups.csv"), index=False)
display(table.round(3))

n = len(table)
p_mw = mannwhitneyu(res[res.adversarial.astype(bool)]["A (active)"], res[~res.adversarial.astype(bool)]["A (active)"], alternative="less").pvalue
ok = c1 == 3 and c2 >= 2 and n == 3
lines = [
    f"C1 mechanism: adversarial models have lower amplification in {c1}/{n} groups (need 3/3) -> {'MET' if c1 == 3 and n == 3 else 'NOT MET'}",
    f"C2 prediction: shifted lambda1 rho >= 0.7 and above clean in {c2}/{n} groups (need 2/3) -> {'MET' if c2 >= 2 else 'NOT MET'}",
    f"pooled Mann-Whitney (A adversarial < non-adversarial), {len(res)} models: p = {p_mw:.4g}",
    f"random-shift control: median A (random) = {res['A (random)'].median():.2f}",
    "PHASE 4 FINDING " + ("CONFIRMED" if ok else "NOT CONFIRMED"),
]
print("\n".join(lines))
open(os.path.join(OUT, "phase5_decision.txt"), "w").write("\n".join(lines))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
groups = sorted(res.group.unique())
colors = dict(zip(groups, ["#3b6ea5", "#d9822b", "#4a9b6e"]))
ax = axes[0]
for i, gname in enumerate(groups):
    g = res[res.group == gname]
    for adv, mk, off in [(False, "o", -0.12), (True, "^", 0.12)]:
        s = g[g.adversarial.astype(bool) == adv]["A (active)"]
        ax.scatter(np.full(len(s), i + off), s, marker=mk, color=colors[gname], s=50,
                   label=("adversarial" if adv else "non-adversarial") if i == 0 else None)
ax.set_yscale("log"); ax.set_xticks(range(len(groups)), [gname.replace(" (phases 1-2)", "\n(phases 1-2)") for gname in groups], fontsize=8)
ax.set(ylabel=r"amplification $A = \lambda_1(C_\delta)/\lambda_1(C)$", title="held-out models"); ax.legend(fontsize=8)
ax = axes[1]
for gname in groups:
    g = res[res.group == gname]
    ax.scatter(g["lambda1 active d4"], g["AS fooling"] * 100, color=colors[gname], s=45, label=gname)
ax.set_xscale("log"); ax.set(xlabel=r"$\lambda_1(C_\delta)$ at $x \pm \delta u_1$", ylabel="AS universal fooling (%)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase5_confirmatory.png"), bbox_inches="tight"); plt.show()